# Implied Volatility Surface -- Pipeline Explorer

Interactive walkthrough of the full pipeline (Modules 1-6). Two ways to run:

1. **Live validation**: pull a snapshot first (`python -m src.data --snapshot`,
   during market hours), then run all cells. The notebook picks up the most
   recent file in `data/snapshots/`.
2. **Demo**: with no snapshot present, a synthetic SPX-like chain is generated
   (known forwards, quote noise, one junk quote, an SPX/SPXW settlement mix)
   so every stage has something real to show.

Design history and rationale live in `README.md` (revision log r1-r5) and
`PROJECT_SPEC.md`. Nothing here changes pipeline behavior; the notebook only
calls the same functions the CLI does.

In [ ]:
import glob, os, sys, logging
from pathlib import Path

# Run from notebooks/ or the project root
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
    os.chdir(root)
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = "notebook"
logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(message)s")

from src.data import (DEFAULT_CONFIG, attach_forwards, clean_chain,
                      load_snapshot, save_snapshot)
from src.iv_solver import check_parity_iv, compute_iv_surface
from src.arbitrage import exclude_flagged, run_arbitrage_checks
from src.svi import calibrate_all_slices
from src.surface import (build_interpolator, check_fitted_calendar,
                         evaluate_svi_grid)
from src.viz import (atm_forward_term_structure, plot_3d_surface,
                     plot_smile_slices, plot_term_structure)

CONFIG = {**DEFAULT_CONFIG}
snapshots = sorted(glob.glob("data/snapshots/*.csv"))
print(f"snapshots found: {len(snapshots)}")

## Stage 0 -- Snapshot

Everything downstream of the fetch runs offline from a snapshot; that is the
reproducibility contract. If none exists, build the synthetic demo chain.

In [ ]:
def make_demo_snapshot():
    """Synthetic chain with the failure modes r5 was built for: quote
    noise, an SPX/SPXW mixed third-Friday expiry with a settlement basis,
    one dead quote, one crossed row."""
    from scipy.stats import norm
    rng = np.random.default_rng(7)
    SPOT, ASOF = 6500.0, pd.Timestamp("2026-07-09 14:00")

    def b76(F, K, DF, T, s, typ):
        d1 = (np.log(F/K) + 0.5*s*s*T) / (s*np.sqrt(T)); d2 = d1 - s*np.sqrt(T)
        c = DF*(F*norm.cdf(d1) - K*norm.cdf(d2))
        return c if typ == "call" else c - DF*(F - K)

    def leg(F, DF, days, exp, strikes, root):
        T = days/365; rows = []
        for K in strikes:
            iv = 0.18 + 0.15*np.log(F/K)
            for typ in ("call", "put"):
                mid = b76(F, K, DF, T, iv, typ)
                sym = f"{root}{pd.Timestamp(exp).strftime('%y%m%d')}" \
                      f"{'C' if typ=='call' else 'P'}{int(K*1000):08d}"
                rows.append(dict(contractSymbol=sym, strike=K, lastPrice=mid,
                                 bid=max(mid-0.25, 0.01), ask=mid+0.25,
                                 volume=500.0, openInterest=1000.0,
                                 impliedVolatility=iv, option_type=typ,
                                 expiry=pd.Timestamp(exp)))
        return pd.DataFrame(rows)

    frames = [
        leg(6510., 0.997, 43, "2026-08-21", np.arange(5900, 7150, 50.), "SPXW"),
        leg(6510., 0.997, 43, "2026-08-21", np.arange(5925, 7175, 50.), "SPX"),
        leg(6525., 0.993, 74, "2026-09-21", np.arange(5900, 7150, 25.), "SPXW"),
        leg(6540., 0.988, 134, "2026-11-20", np.arange(5900, 7150, 25.), "SPXW"),
    ]
    frames[1][["bid", "ask"]] += 1.4               # settlement basis
    chain = pd.concat(frames, ignore_index=True)
    dead = (chain["strike"] == 6275.) & (chain["option_type"] == "put") \
           & (chain["expiry"] == pd.Timestamp("2026-11-20"))
    chain.loc[dead, ["bid", "ask"]] += 70.0        # junk quote
    n = rng.uniform(-0.3, 0.3, len(chain))
    chain["bid"] = np.maximum(chain["bid"] + n - 0.15, 0.01)
    chain["ask"] = chain["ask"] + n + 0.15
    return save_snapshot(chain, SPOT, "^SPX", asof=ASOF)

snap_path = snapshots[-1] if snapshots else make_demo_snapshot()
chain, spot, asof = load_snapshot(snap_path)
print(f"{snap_path}\nspot={spot:.2f}  asof={asof}  rows={len(chain)}  "
      f"expiries={chain['expiry'].nunique()}")

## Stage 1 -- Cleaning and implied forwards

Watch the per-filter drop counts (root filter and crossed-quote guard are the
r5 additions). Then the parity regression per expiry: the implied financing
minus the FRED diagnostic is the market's dividend/borrow pricing -- for SPX
expect implied financing BELOW the risk-free diagnostic by roughly the
dividend yield.

In [ ]:
clean = clean_chain(chain, spot, CONFIG, asof=asof)
both_sides = attach_forwards(clean, spot, CONFIG, otm_only=False)
parity_breaches = check_parity_iv(both_sides, CONFIG, tol=0.01)
print(f"\ncall-vs-put IV breaches > 1 vol pt: {len(parity_breaches)}")
otm = attach_forwards(clean, spot, CONFIG)
fwd_table = (otm.groupby("expiry")[["T", "F", "DF", "fwd_fallback"]]
             .first().assign(implied_financing=lambda d: -np.log(d["DF"])/d["T"]))
fwd_table

In [ ]:
# Visual: the parity regression for the nearest expiry.
import matplotlib.pyplot as plt
exp0 = sorted(clean["expiry"].unique())[0]
g = clean[clean["expiry"] == exp0]
calls = g[g["option_type"]=="call"][["strike","mid_price"]]
puts = g[g["option_type"]=="put"][["strike","mid_price"]]
pairs = calls.merge(puts, on="strike", suffixes=("_c","_p"))
pairs = pairs[np.abs(pairs["strike"]/spot - 1) <= CONFIG["FWD_BAND"]]
F0 = float(otm.loc[otm["expiry"]==exp0, "F"].iloc[0])
DF0 = float(otm.loc[otm["expiry"]==exp0, "DF"].iloc[0])
fig, ax = plt.subplots(figsize=(7,4))
ax.scatter(pairs["strike"], pairs["mid_price_c"]-pairs["mid_price_p"], s=14)
ks = np.array([pairs["strike"].min(), pairs["strike"].max()])
ax.plot(ks, DF0*(F0-ks), lw=1.5,
        label=f"fit: DF={DF0:.4f}, F={F0:.1f}")
ax.set_xlabel("strike"); ax.set_ylabel("C - P (mid)")
ax.set_title(f"Put-call parity regression, {pd.Timestamp(exp0).date()}")
ax.legend(); ax.grid(alpha=0.3); plt.show()

## Stage 2 -- IV inversion (Black-76, MK-seeded NR + Brent)

The solver split matters: near-100% NR is healthy; a large Brent share means
low-vega junk survived cleaning.

In [ ]:
surf_df = compute_iv_surface(otm, CONFIG)
fig, ax = plt.subplots(figsize=(8,4.5))
for exp, g in surf_df.groupby("expiry"):
    ax.scatter(g["k"], g["iv"]*100, s=8, alpha=0.6,
               label=str(pd.Timestamp(exp).date()))
ax.set_xlabel("k = log(K/F)"); ax.set_ylabel("IV (%)")
ax.set_title("Raw inverted smiles (pre-fit)"); ax.legend(); ax.grid(alpha=0.3)
plt.show()

## Stage 3 -- Static-arbitrage checks (vertical -> butterfly -> calendar)

Vertical runs first and its executable offenders (junk quotes) are removed
before the other checks. Executable flags are the exclusion tier; mid-only
flags are data-quality diagnostics inside the bid-ask.

In [ ]:
vert, bf, cal = run_arbitrage_checks(surf_df, CONFIG | {})
offenders = vert.dropna(subset=["offender"]) if not vert.empty else vert
if not offenders.empty:
    print("attributed junk-quote strikes:")
    display(offenders[["expiry","K_low","K_high","viol_type",
                       "exec_amount","offender"]])
fit_input = exclude_flagged(surf_df, vert, bf, cal)
print(f"fit input: {len(fit_input)}/{len(surf_df)} rows")

## Stage 4 -- SVI calibration

Per-slice DE global search + L-BFGS-B polish. Watch the warnings: RMSE above
0.005, Lee wing slopes above 2, bound-railed parameters, and a vertex `m`
outside the observed k range all mean 'do not trust the wings'.

In [ ]:
params = calibrate_all_slices(fit_input, None)
params

## Stage 5 -- Surface construction

Fitted-slice calendar containment first (linear-in-total-variance time
interpolation is only arbitrage-free where it holds), then the query API.

In [ ]:
xc = check_fitted_calendar(params)
print("fitted-slice calendar:", "CLEAN" if xc.empty else "")
if not xc.empty: display(xc)
interp = build_interpolator(params, spot)
T_grid = params["T"].to_numpy()
Tq = float(np.mean(T_grid[:2])) if len(T_grid) > 1 else float(T_grid[0])
print(f"sample queries at T={Tq:.3f}:")
for K in (0.95*spot, spot, 1.05*spot):
    print(f"  K={K:8.1f}: iv={interp(Tq, K):.4f}")

## Stage 6 -- Rendering

The moneyness grid is restricted to roughly the observed range, consistent
with the identifiability caveats: wings beyond the data are not rendered.

In [ ]:
m_lo = max(0.85, float((fit_input['strike']/spot).min()))
m_hi = min(1.15, float((fit_input['strike']/spot).max()))
m_grid = np.linspace(m_lo, m_hi, 120)
iv_matrix = evaluate_svi_grid(params, m_grid, spot)

import plotly.graph_objects as go
fig = go.Figure(data=[go.Surface(x=m_grid, y=params["T"].to_numpy()*365,
                                 z=iv_matrix*100, colorscale="RdYlGn_r",
                                 colorbar=dict(title="IV (%)"))])
fig.update_layout(title=f"IV surface -- spot {spot:.2f} @ {asof}",
                  scene=dict(xaxis_title="Moneyness (K/S)",
                             yaxis_title="Days to expiry",
                             zaxis_title="Implied vol (%)"),
                  margin=dict(l=10, r=10, t=50, b=10), height=550)
fig.show()

In [ ]:
smile_paths = plot_smile_slices(fit_input, params, "outputs/smiles")
ts_path = plot_term_structure(fit_input, "outputs/term_structure.png")
html_path = plot_3d_surface(iv_matrix, m_grid, params["T"].to_numpy(),
                            "outputs/vol_surface.html",
                            title=f"IV surface -- spot {spot:.2f} @ {asof}")
ts = atm_forward_term_structure(fit_input)
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(ts["days_to_expiry"], ts["iv"]*100, marker="o")
ax.set_xlabel("days to expiry"); ax.set_ylabel("ATM-forward IV (%)")
ax.set_title("Term structure (k = 0)"); ax.grid(alpha=0.3); plt.show()
print(f"written: {html_path}, {len(smile_paths)} smiles, {ts_path}")

## Validation checklist (live snapshots)

- Root filter: drop count at `clean_chain [option root]` should be nonzero on
  third-Friday expiries; executable butterfly flags should NOT cluster there.
- Crossed-quote guard: `ask>=bid>0` drop count small but nonzero.
- Parity regression: `resid_rms` under ~$0.50 on liquid expiries; implied
  financing below the FRED diagnostic by roughly a dividend yield.
- Solver: NR share near 100%, zero failed inversions.
- Arbitrage: executable butterflies near zero; junk strikes appear as
  attributed vertical offenders instead.
- SVI: no RMSE/Lee/bound-rail/vertex warnings on liquid slices.
- Fitted calendar: clean on the observed k range.

Anything outside these bands: stop, inspect `outputs/arbitrage_flags.csv`,
and re-read the r5 revision-log entry before trusting the surface.